# April experiment — INTC & NVDA, fully separated, original-main() defaults

Runs the experiment from `LearningKraus.py`'s original `main()` (as committed on the
**locked `main` baseline**, tag `baseline`) end to end, from raw data:

1. **Configs** — every April-2025 trading day on disk (21), one config per symbol,
   `instrument_filter: true` so INTC and NVDA are **entirely separated** (own filtered
   event stream, own feature cache, own distributions, own models — they share nothing
   but the read-only raw files).
2. **Distributions** — featurize → encode → `SEQ_DISTR_*`/`CLS_DISTR_*` per symbol.
3. **Training** — 2 symbols × 3 predictors = **6 models**, each a verbatim re-run of the
   original `main()` body. Per model you get the **2 result files + 4 charts**
   (6×2 = 12 files, 6×4 = 24 images), shown inline below as each model completes —
   forward each bundle as soon as it appears.

Every default in the next cell is the value from the baseline `main()` — change them
only to deviate from the original experiment. Training is unseeded (like the original)
unless you set `SEED`.

*Long runs over SSH: JupyterLab keeps the kernel alive if the browser disconnects; from a
plain terminal you can run the same thing detached with `scripts/run_april.py` under
`nohup`/`tmux`.*

In [ ]:
# ---- parameters (defaults = LearningKraus.main() on the locked baseline) ----
SYMBOLS = ["NVDA", "INTC"]                        # per boss: both, separated
PREDICTORS = ["tvi_n", "obi_L1", "ofi_L1_n_norm"]  # features_list[1..3]

EPOCHS = 3000        # main(): epochs=3000        (use e.g. 50 for a smoke run)
N_QUBITS = 3         # main(): n_qubits = 3
SEED = None          # main() is unseeded; set an int for reproducible runs
# (batch_size=6*512, lr=1e-3, adam, nll_seq, learn_rho0=True, max_seq_len=6,
#  min_seq_prob=0.0, m=16, num_workers=8, device=cuda-else-cpu are fixed
#  inside the harness at exactly the original values)

DATA_DIR = None      # None = auto-discover data/NVDA_INTC in/near the repo.
                     # On the compute box set the absolute path, e.g.
                     # "/home/you/src/time_series_prediction-dev/data/NVDA_INTC"
WORKERS = 4          # day-parallel featurize workers; 0 = one per core (Linux box)
SKIP_DISTRIBUTIONS = False   # True to retrain on existing April distributions
RUN_ENSEMBLE = False # True: also build the ENS_TD_* ensemble tables per symbol
                     # (colleague's experiment: 5 lengths x 5 classes = 25
                     #  files/symbol; defaults in the ensemble config section)

In [ ]:
import json, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import find_data_dir, april_dates, detect_pattern, make_config
from pipeline.config import RunConfig

data_dir = pathlib.Path(DATA_DIR).resolve() if DATA_DIR else find_data_dir()
assert data_dir.is_dir(), f"data dir not found: {data_dir}"
pattern = detect_pattern(data_dir)     # .dbn.zst or plain .dbn — both read fine
dates = april_dates(data_dir, pattern)
print(f"data: {data_dir}  (pattern: {pattern})")
print(f"April days: {len(dates)} ({dates[0]}..{dates[-1]})")

configs = {s: make_config(s, data_dir, dates, WORKERS, PREDICTORS, pattern)
           for s in SYMBOLS}

### Stage 2 — per-symbol distributions
One decode+featurize per (symbol, day), day-parallel; outputs land in
`outputs/april/{SYMBOL}/`. Skipped if `SKIP_DISTRIBUTIONS = True`.

In [ ]:
if not SKIP_DISTRIBUTIONS:
    from pipeline.runner import run
    for symbol, cfg_path in configs.items():
        print(f"=== distributions: {symbol} ({len(dates)} days x {len(PREDICTORS)} predictors) ===")
        t0 = time.time()
        run(RunConfig.load(cfg_path), run_id=f"april-{symbol}")
        print(f"{symbol} done in {time.time()-t0:.0f}s -> outputs/april/{symbol}/\n")
else:
    print("skipped (SKIP_DISTRIBUTIONS=True)")

### Stage 2b (optional) — ensemble training tables (`ENS_TD_*`)
The colleague's fixed-length multi-channel experiment: 11 timestamp-aligned
bivariate channels, 5 sequence lengths × 5 class definitions → 25 pickles per
symbol under `outputs/april/{SYMBOL}/ensemble/`. Counting math is his code
verbatim (byte-equivalence verified by `tests/verify_ensemble_reference.py`);
featurize is served from the same per-symbol cache as stage 2, so this adds
roughly 30 min per symbol warm. Enable with `RUN_ENSEMBLE = True`.

In [ ]:
if RUN_ENSEMBLE:
    from pipeline.ensemble import run_ensemble
    for symbol, cfg_path in configs.items():
        print(f"=== ensemble tables: {symbol} ===")
        t0 = time.time()
        outputs = run_ensemble(RunConfig.load(cfg_path),
                               run_id=f"april-ensemble-{symbol}")
        print(f"{symbol}: {len(outputs)} ENS_TD files in "
              f"{time.time()-t0:.0f}s -> outputs/april/{symbol}/ensemble/\n")
else:
    print("skipped (RUN_ENSEMBLE=False)")

### Stage 3 — the 6 models (send-as-you-go)
Each iteration is the original `main()` run for one (symbol, predictor). As each model
finishes, its **READY TO SEND** bundle is printed and the 4 charts render inline.
Progress persists in `outputs/april/april_summary.json` — safe to interrupt and re-run
with `SKIP_DISTRIBUTIONS = True`.

In [ ]:
from IPython.display import Image, display
from train_kraus_baseline import run_one

summary_path = ROOT / "outputs" / "april" / "april_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
results = []
combos = [(s, p) for s in SYMBOLS for p in PREDICTORS]

for i, (symbol, predictor) in enumerate(combos, 1):
    distr_dir = ROOT / "outputs" / "april" / symbol
    out_dir = distr_dir / "models"
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== MODEL {i}/{len(combos)}: {symbol} x {predictor} "
          f"(epochs={EPOCHS}, {N_QUBITS}q) ===")
    r = run_one(predictor, distr_dir, out_dir, EPOCHS, SEED,
                symbol=symbol, n_qubits=N_QUBITS)
    results.append(r)
    summary_path.write_text(json.dumps(results, indent=1))

    print(f"\n>>> MODEL {i}/{len(combos)} COMPLETE — READY TO SEND:")
    print(f"    result file 1: {r['model_pickle']}")
    print(f"    result file 2: {r['weights']}")
    for png in r["plots"]:
        print(f"    image:         {png}")
    print(f"    cost={r['final_cost_weighted_mse']:.3e}  ({r['train_seconds']:.0f}s)")
    for png in r["plots"]:
        display(Image(filename=png))

print(f"\nAll {len(combos)} models done. Summary: {summary_path}")

### Where everything lands
```
configs/april_nvda.yaml, april_intc.yaml      the experiment definitions
outputs/april/{SYMBOL}/SEQ_DISTR_*, CLS_*     per-symbol distributions
outputs/april/{SYMBOL}/feature_cache/         per-symbol featurized days
outputs/april/{SYMBOL}/ensemble/ENS_TD_*      25 ensemble tables (stage 2b)
outputs/april/{SYMBOL}/models/MOD*, WGHTS_*   2 result files per model
outputs/april/{SYMBOL}/models/*_{n}q_1..4.png 4 charts per model
outputs/april/april_summary.json              cost + timing per model
```